# **Tracking**
Use this notebook to generate football tracking data from match footage via the trained Pitch Detection Model and Player Detection Model we trained.

---

In [2]:
!git clone https://github.com/JohnComonitski/FootballTrackingDataGeneration.git /content/FootballTrackingDataGeneration

fatal: destination path '/content/FootballTrackingDataGeneration' already exists and is not an empty directory.


In [3]:
%cd /content/FootballTrackingDataGeneration

/content/FootballTrackingDataGeneration


In [4]:
!pip install -q "inference==1.7.3" "supervision>=0.29,<0.30" python-dotenv tqdm
!pip install -q "git+https://github.com/roboflow/sports.git"

ERROR: Cannot install inference==1.7.3 and supervision<0.30 and >=0.29 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
  Preparing metadata (setup.py) ... done


In [5]:
from getpass import getpass
from pathlib import Path

api_key = getpass("Roboflow API key: ")

Path("/content/FootballTrackingDataGeneration/env").mkdir(exist_ok=True)

Path("/content/FootballTrackingDataGeneration/env/keys.env").write_text(
    f"ROBOFLOW_API={api_key}\n",
    encoding="utf-8"
)

print("API key saved")

Roboflow API key: ··········
API key saved


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os

os.makedirs(
    "/content/FootballTrackingDataGeneration/track/footage",
    exist_ok=True
)

In [8]:
!ffmpeg -y \
-i "/content/drive/MyDrive/FootballAI/match.webm" \
-c:v libx264 \
-preset fast \
-crf 23 \
-c:a aac \
"/content/FootballTrackingDataGeneration/track/footage/match.mp4"

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [9]:
%pip uninstall -y roboflow

%pip install "inference==1.7.3" "supervision==0.29.1" python-dotenv tqdm

%pip install "git+https://github.com/roboflow/sports.git"

  Using cached supervision-0.29.1-py3-none-any.whl.metadata (14 kB)
INFO: pip is looking at multiple versions of inference to determine which version is compatible with other requirements. This could take a while.
ERROR: Cannot install inference==1.7.3 and supervision==0.29.1 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested supervision==0.29.1
    inference 1.7.3 depends on supervision<0.31.0 and >=0.30.6

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict

ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
  Cloning https://github.com/roboflow/sports.git to /tmp/pip-req-build-hl209k1b
  Running command git clone --filter=blob:none --quiet https://github.com/roboflow/sports.git /tmp/pip-req-build-hl209k1b
  Resolved htt

In [10]:
import sys

PYTHON = sys.executable

!$PYTHON -m pip install --no-cache-dir --upgrade \
"inference==1.7.3" \
"supervision==0.30.8"

In [11]:
import sys

!{sys.executable} -m pip show inference
!{sys.executable} -m pip show supervision

Name: inference
Version: 1.7.3
Summary: With no prior knowledge of machine learning or device-specific deployment, you can deploy a computer vision model to a range of devices and environments using Roboflow Inference.
Home-page: https://github.com/roboflow/inference
Author: Roboflow
Author-email: help@roboflow.com
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: aiohttp, aiortc, anthropic, APScheduler, av, backoff, boto3, botocore, cachetools, click, cython, dataclasses-json, docker, fastapi, filelock, httpx, inference-models, inference-sdk, msgpack, networkx, numpy, nvidia-ml-py, onnxruntime, onvif-zeep-async, openai, opencv-contrib-python, opencv-python, packaging, pandas, pillow, prometheus-fastapi-instrumentator, psutil, py-cpuinfo, pybase64, pydantic, pydantic-settings, pydot, pytest, python-dotenv, PyYAML, qrcode, redis, requests, requests-toolbelt, rich, roboflow-workflows, scikit-image, setuptools, shapely, simple-pid, slack-sdk, streamvision, structlog, s

In [12]:
from inference import get_model
import supervision as sv
from sports.common.team import TeamClassifier
from sports.configs.soccer import SoccerPitchConfiguration

print("ВСЁ УСТАНОВЛЕНО")
print("supervision:", sv.__version__)

ВСЁ УСТАНОВЛЕНО
supervision: 0.30.8


In [13]:
from PIL import Image, ImageDraw, ImageFont
from inference import get_model
import supervision as sv

print("PIL OK")
print("INFERENCE OK")
print("Supervision:", sv.__version__)

PIL OK
INFERENCE OK
Supervision: 0.30.8


In [14]:
import os

path = "/content/FootballTrackingDataGeneration/track/footage/match.mp4"

print("Найден:", os.path.exists(path))
print("Размер:", os.path.getsize(path) / 1024 / 1024, "MB")

Найден: True
Размер: 43.37637519836426 MB


In [2]:
import torch
import onnxruntime as ort
import supervision as sv
from inference import get_model

print("Torch CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "НЕТ GPU")
print("ONNX providers:", ort.get_available_providers())
print("Supervision:", sv.__version__)
print("Inference import: OK")

/usr/local/lib/python3.13/dist-packages/inference/core/env.py:345: InferenceDeprecationWarning: CORE_MODEL_GAZE_ENABLED is True: POST /gaze/gaze_detection is registered as a deprecation stub returning HTTP 410 Gone. The stub and this flag will be removed end of Q2 2026. Set CORE_MODEL_GAZE_ENABLED=False to disable it now.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[transformers] `DepthProImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `DepthProImageProcessor` instead.


Torch CUDA: True
GPU: Tesla T4
ONNX providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Supervision: 0.30.8
Inference import: OK


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/inference/models/utils.py:763: ModelDependencyMissing: Your `inference` configuration does not support YoloWorld model. Use pip install 'inference[yolo-world]' to install missing requirements.To suppress this warning, set CORE_MODEL_YOLO_WORLD_ENABLED to False.
  warnings.warn(


In [16]:
%cd /content/FootballTrackingDataGeneration/track

/content/FootballTrackingDataGeneration/track


In [17]:
!ffmpeg -y -ss 00:01:00 -i "./footage/match.mp4" -t 30 -c copy "./footage/test30.mp4"

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [18]:
import os
print(os.path.exists("./footage/test30.mp4"))

True


## Preparation

**Configurations**

In [19]:
GENERATE_VIDEO = 1
TRACK_TEAMS = 1

**Imports**

In [5]:
from inference import get_model
#from ultralytics import YOLO
import supervision as sv
from tqdm import tqdm
#import pandas as pd
import numpy as np
from sports.common.team import TeamClassifier
#from sports.annotators.soccer import draw_pitch, draw_points_on_pitch
from sports.configs.soccer import SoccerPitchConfiguration
from sports.common.view import ViewTransformer
#from supervision.draw.utils import draw_image
#from typing import List, Union
#from collections import deque

In [6]:
from dotenv import load_dotenv
import os
os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[CUDAExecutionProvider]"

**Roboflow API Key**

Before running this block, make sure your Roboflow API keys are stored in `./env/keys.env` as specifed in the [Getting Started](https://github.com/JohnComonitski/FootballTrackingDataGeneration?tab=readme-ov-file#getting-started) section of this repo.

In [7]:
load_dotenv("./../env/keys.env")

# Access the variables
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API")

---

## Getting Your Models

Repalce the values of `PITCH_DETECTION_MODEL_ID` and `PLAYER_DETECTION_MODEL_ID` with the Roboflow Project ID's and Model Version Numbers which you created in the training notebook

In [23]:
PITCH_DETECTION_MODEL_ID = "football-field-detection-f07vi-yukgc/1"
PITCH_DETECTION_MODEL = get_model(model_id=PITCH_DETECTION_MODEL_ID, api_key=ROBOFLOW_API_KEY)

Output()

Model requested device cuda, but the onnxruntime session runs only CPU-bound execution providers (CPUExecutionProvider) - falling back to cpu tensors for this model.


In [24]:
PLAYER_DETECTION_MODEL_ID = "football-players-detection-3zvbc-btky1/1"
PLAYER_DETECTION_MODEL = get_model(model_id=PLAYER_DETECTION_MODEL_ID, api_key=ROBOFLOW_API_KEY)

Output()

Model requested device cuda, but the onnxruntime session runs only CPU-bound execution providers (CPUExecutionProvider) - falling back to cpu tensors for this model.


---

## Preparing Your Data Source

Update `VIDEO_FILE` with the match footage you want to track

In [25]:
VIDEO_FILE = "test30"
SOURCE_VIDEO_PATH = './footage/' + VIDEO_FILE + ".mp4"

---

## Object Detection

**Classes**

In [26]:
objects = {
    "ball" : 0,
    "goalkeeper" : 1,
    "player" : 2,
    "referee" :3
}

**Annotations**

In [27]:
ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'),
    base=25,
    height=21,
    outline_thickness=1
)
box_annotator = sv.BoxAnnotator(
    color=sv.ColorPalette.from_hex(['#FF8C00', '#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)

**Identify Goal Keeper**

In [35]:
def resolve_goalkeepers_team_id(players, goalkeepers):
    if len(goalkeepers) == 0:
        return np.array([], dtype=int)

    goalkeepers_xy = goalkeepers.get_anchors_coordinates(
        sv.Position.BOTTOM_CENTER
    )

    if len(players) == 0 or players.class_id is None:
        return np.zeros(len(goalkeepers), dtype=int)

    players_xy = players.get_anchors_coordinates(
        sv.Position.BOTTOM_CENTER
    )

    team0 = players_xy[players.class_id == 0]
    team1 = players_xy[players.class_id == 1]

    # Если в конкретном кадре видна только одна команда
    if len(team0) == 0 and len(team1) == 0:
        return np.zeros(len(goalkeepers), dtype=int)

    if len(team0) == 0:
        return np.ones(len(goalkeepers), dtype=int)

    if len(team1) == 0:
        return np.zeros(len(goalkeepers), dtype=int)

    team_0_centroid = team0.mean(axis=0)
    team_1_centroid = team1.mean(axis=0)

    goalkeeper_team_ids = []

    for goalkeeper_xy in goalkeepers_xy:
        dist_0 = np.linalg.norm(
            goalkeeper_xy - team_0_centroid
        )
        dist_1 = np.linalg.norm(
            goalkeeper_xy - team_1_centroid
        )

        goalkeeper_team_ids.append(
            0 if dist_0 < dist_1 else 1
        )

    return np.array(goalkeeper_team_ids, dtype=int)

**Get Detections**

In [41]:
LAST_TRANSFORMER = None
LAST_TRANSFORMER_AGE = 999


def get_detections(
    frame,
    detections,
    key_points,
    tracker,
    team_classifier
):
    global LAST_TRANSFORMER
    global LAST_TRANSFORMER_AGE

    CONFIG = SoccerPitchConfiguration()

    # -----------------------------
    # BALL
    # -----------------------------
    ball_detections = detections[
        detections.class_id == objects["ball"]
    ]

    if len(ball_detections) > 0:
        ball_detections.xyxy = sv.pad_boxes(
            xyxy=ball_detections.xyxy,
            px=10
        )

    # -----------------------------
    # PLAYERS / GK / REFEREES
    # -----------------------------
    all_detections = detections[
        detections.class_id != objects["ball"]
    ]

    all_detections = all_detections.with_nms(
        threshold=0.5,
        class_agnostic=True
    )

    all_detections = tracker.update_with_detections(
        detections=all_detections
    )

    goalkeepers_detections = all_detections[
        all_detections.class_id == objects["goalkeeper"]
    ]

    players_detections = all_detections[
        all_detections.class_id == objects["player"]
    ]

    # -----------------------------
    # TEAM CLASSIFICATION
    # -----------------------------
    if team_classifier is not None and len(players_detections) > 0:

        players_crops = [
            sv.crop_image(frame, xyxy)
            for xyxy in players_detections.xyxy
        ]

        if len(players_crops) > 0:
            players_detections.class_id = (
                team_classifier.predict(players_crops)
            )

    if len(goalkeepers_detections) > 0:
        goalkeepers_detections.class_id = (
            resolve_goalkeepers_team_id(
                players_detections,
                goalkeepers_detections
            )
        )

    # ============================================================
    # PITCH HOMOGRAPHY
    # ============================================================

    new_transformer = None

    if (
        key_points is not None
        and hasattr(key_points, "xy")
        and key_points.xy is not None
        and len(key_points.xy) > 0
        and key_points.xy.shape[1] > 0
    ):

        xy = key_points.xy[0]

        confidence = getattr(
            key_points,
            "keypoint_confidence",
            None
        )

        # Новая версия supervision
        if (
            confidence is not None
            and len(confidence) > 0
        ):
            valid = confidence[0] > 0.3

        # Если confidence отсутствует —
        # используем реально найденные координаты
        else:
            valid = np.logical_and(
                np.isfinite(xy).all(axis=1),
                np.any(xy != 0, axis=1)
            )

        frame_reference_points = xy[valid]

        vertices = np.array(
            CONFIG.vertices,
            dtype=np.float32
        )

        # На случай разного количества keypoints
        valid_indices = np.where(valid)[0]
        valid_indices = valid_indices[
            valid_indices < len(vertices)
        ]

        frame_reference_points = xy[
            valid_indices
        ]

        pitch_reference_points = vertices[
            valid_indices
        ]

        # Для homography минимум 4 точки
        if len(frame_reference_points) >= 4:
            try:
                new_transformer = ViewTransformer(
                    source=frame_reference_points,
                    target=pitch_reference_points
                )
            except Exception:
                new_transformer = None

    # Если в этом кадре поле распознано —
    # обновляем homography
    if new_transformer is not None:
        LAST_TRANSFORMER = new_transformer
        LAST_TRANSFORMER_AGE = 0
    else:
        LAST_TRANSFORMER_AGE += 1

    # Небольшой запас, если модель пропустила несколько кадров подряд
    transformer = None

    if (
        LAST_TRANSFORMER is not None
        and LAST_TRANSFORMER_AGE <= 30
    ):
        transformer = LAST_TRANSFORMER

    # -----------------------------
    # PIXEL ANCHORS
    # -----------------------------
    frame_ball_xy = (
        ball_detections.get_anchors_coordinates(
            sv.Position.BOTTOM_CENTER
        )
    )

    frame_goalkeepers_xy = (
        goalkeepers_detections.get_anchors_coordinates(
            sv.Position.BOTTOM_CENTER
        )
    )

    frame_players_xy = (
        players_detections.get_anchors_coordinates(
            sv.Position.BOTTOM_CENTER
        )
    )

    # -----------------------------
    # CONVERT TO PITCH
    # -----------------------------
    if transformer is not None:

        pitch_ball_xy = transformer.transform_points(
            points=frame_ball_xy
        )

        pitch_goalkeepers_xy = transformer.transform_points(
            points=frame_goalkeepers_xy
        )

        pitch_players_xy = transformer.transform_points(
            points=frame_players_xy
        )

    else:

        # Не придумываем координаты.
        # Эти строки потом просто будут проигнорированы.
        pitch_ball_xy = np.full(
            (len(frame_ball_xy), 2),
            np.nan
        )

        pitch_goalkeepers_xy = np.full(
            (len(frame_goalkeepers_xy), 2),
            np.nan
        )

        pitch_players_xy = np.full(
            (len(frame_players_xy), 2),
            np.nan
        )

    ball_detections.data[
        "pitch_xy"
    ] = pitch_ball_xy

    goalkeepers_detections.data[
        "pitch_xy"
    ] = pitch_goalkeepers_xy

    players_detections.data[
        "pitch_xy"
    ] = pitch_players_xy

    # -----------------------------
    # MERGE
    # -----------------------------
    all_detections = sv.Detections.merge([
        players_detections,
        goalkeepers_detections
    ])

    return all_detections, ball_detections

**Identify Teams**

In [37]:
def generate_team_model(video, PLAYER_DETECTION_MODEL):
  STRIDE = 30

  frame_generator = sv.get_video_frames_generator(
      source_path=SOURCE_VIDEO_PATH, stride=STRIDE)

  crops = []
  for frame in tqdm(frame_generator, desc='collecting crops'):
      result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
      detections = sv.Detections.from_inference(result)
      players_crops = [sv.crop_image(frame, xyxy) for xyxy in detections.xyxy]
      crops += players_crops

  #team_classifier = TeamClassifier(device="cuda")
  team_classifier = TeamClassifier(device="cuda")
  team_classifier.fit(crops)

  return team_classifier

**Output Results**

In [38]:
def save_tracking_results(players, ball, frames):
  csv = "Frame,Object,Object ID,Team,X1,Y1,X1,X2,X_Pitch,Y_Pitch\n"
  for frame in range(1, frames):
    for player in players:
      player_data = players[player]
      if str(frame) in player_data:
        csv += str(frame) + ",player," + str(player) + "," + str(player_data[str(frame)]["Team"]) + "," + str(player_data[str(frame)]["X1"]) + "," + str(player_data[str(frame)]["Y1"]) + ","  + str(player_data[str(frame)]["X2"]) + "," + str(player_data[str(frame)]["Y2"]) + "," + str(player_data[str(frame)]["X_Pitch"]) + "," + str(player_data[str(frame)]["Y_Pitch"]) + "\n"
    if str(frame) in ball:
      csv += str(frame) + ",ball,,," + str(ball[str(frame)]["X1"]) + "," + str(ball[str(frame)]["Y1"]) + "," + str(ball[str(frame)]["X2"]) + "," + str(ball[str(frame)]["Y2"]) + "," + str(ball[str(frame)]["X_Pitch"]) + "," + str(ball[str(frame)]["Y_Pitch"]) + "\n"

  with open("./output/" + VIDEO_FILE + ".csv", "w") as file:
    file.write(csv)

**Tracking**

In [42]:
CONFIG = SoccerPitchConfiguration()
tracking_results = ""

#Get Video
frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)

tracker = sv.ByteTrack()
tracker.reset()

In [43]:
team_classifier = None
if(TRACK_TEAMS):
  team_classifier = generate_team_model(SOURCE_VIDEO_PATH, PLAYER_DETECTION_MODEL)

collecting crops: 15it [02:53, 11.58s/it]


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

Embedding extraction: 6it [01:21, 13.66s/it]


In [48]:
%pip uninstall -y inference inference-gpu onnxruntime onnxruntime-gpu

%pip install -q "inference-gpu==1.7.3" "supervision==0.30.8"

%pip install -q "git+https://github.com/roboflow/sports.git"

Found existing installation: inference 1.7.3
Uninstalling inference-1.7.3:
  Successfully uninstalled inference-1.7.3
Found existing installation: onnxruntime 1.21.1
Uninstalling onnxruntime-1.21.1:
  Successfully uninstalled onnxruntime-1.21.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 36.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.8/280.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.9/26.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 12.2 MB/

In [3]:
import time
import supervision as sv

test_generator = sv.get_video_frames_generator(
    source_path=SOURCE_VIDEO_PATH
)

times = []

for i, frame in enumerate(test_generator):
    if i >= 10:
        break

    start = time.perf_counter()

    PLAYER_DETECTION_MODEL.infer(
        frame,
        confidence=0.3
    )[0]

    PITCH_DETECTION_MODEL.infer(
        frame,
        confidence=0.3
    )[0]

    elapsed = time.perf_counter() - start
    times.append(elapsed)

    print(f"Кадр {i+1}: {elapsed:.2f} сек")

print()
print("Среднее:", sum(times) / len(times), "сек/кадр")

In [47]:
import os
import numpy as np
import supervision as sv
from tqdm import tqdm

CONFIG = SoccerPitchConfiguration()

os.makedirs("./output", exist_ok=True)

frame_generator = sv.get_video_frames_generator(
    source_path=SOURCE_VIDEO_PATH
)

tracker = sv.ByteTrack()
tracker.reset()

frame_number = 1
players = {}
ball = {}

frames_total = 0
frames_with_players = 0
frames_with_pitch = 0
frames_with_ball = 0

video_info = sv.VideoInfo.from_video_path(
    video_path=SOURCE_VIDEO_PATH
)

with sv.VideoSink(
    target_path="./output.mp4",
    video_info=video_info
) as sink:

    for frame in tqdm(
        frame_generator,
        desc="Collecting Tracking Data..."
    ):

        frames_total += 1

        # -------------------------
        # PLAYER DETECTION
        # -------------------------
        player_result = PLAYER_DETECTION_MODEL.infer(
            frame,
            confidence=0.3
        )[0]

        detections = sv.Detections.from_inference(
            player_result
        )

        # -------------------------
        # PITCH DETECTION
        # -------------------------
        pitch_result = PITCH_DETECTION_MODEL.infer(
            frame,
            confidence=0.3
        )[0]

        key_points = sv.KeyPoints.from_inference(
            pitch_result
        )

        # -------------------------
        # TRACK / PROJECT
        # -------------------------
        all_detections, ball_detections = get_detections(
            frame,
            detections,
            key_points,
            tracker,
            team_classifier
        )

        n_players = len(all_detections)
        n_ball = len(ball_detections)

        if n_players > 0:
            frames_with_players += 1

        # -------------------------
        # SAFE TRACK IDS
        # -------------------------
        if (
            all_detections.tracker_id is not None
            and len(all_detections.tracker_id) == n_players
        ):
            object_ids = all_detections.tracker_id
        else:
            object_ids = np.arange(n_players)

        # -------------------------
        # SAFE TEAM IDS
        # -------------------------
        if (
            all_detections.class_id is not None
            and len(all_detections.class_id) == n_players
        ):
            team_ids = all_detections.class_id
        else:
            team_ids = np.full(
                n_players,
                -1,
                dtype=int
            )

        # -------------------------
        # SAFE OBJECT TYPES
        # -------------------------
        object_types = all_detections.data.get(
            "class_name",
            np.array(
                ["player"] * n_players,
                dtype=object
            )
        )

        # -------------------------
        # SAFE PITCH COORDINATES
        # -------------------------
        pitch_xys = all_detections.data.get(
            "pitch_xy",
            np.full(
                (n_players, 2),
                np.nan,
                dtype=float
            )
        )

        ball_pitch_xys = ball_detections.data.get(
            "pitch_xy",
            np.full(
                (n_ball, 2),
                np.nan,
                dtype=float
            )
        )

        if (
            len(pitch_xys) > 0
            and np.isfinite(pitch_xys).any()
        ):
            frames_with_pitch += 1

        # -------------------------
        # SAVE PLAYERS
        # -------------------------
        for idx, xyxy in enumerate(
            all_detections.xyxy
        ):

            object_id = str(
                int(object_ids[idx])
            )

            if object_id not in players:
                players[object_id] = {}

            pitch_x = np.nan
            pitch_y = np.nan

            if idx < len(pitch_xys):
                pitch_x = float(
                    pitch_xys[idx][0]
                )
                pitch_y = float(
                    pitch_xys[idx][1]
                )

            team_id = int(
                team_ids[idx]
            )

            players[object_id][str(frame_number)] = {
                "Object Type": (
                    str(object_types[idx])
                    if idx < len(object_types)
                    else "player"
                ),
                "Team": team_id,
                "X1": float(xyxy[0]),
                "Y1": float(xyxy[1]),
                "X2": float(xyxy[2]),
                "Y2": float(xyxy[3]),
                "X_Pitch": pitch_x,
                "Y_Pitch": pitch_y,
                "Y_MPLSoccer": (
                    pitch_y / CONFIG.width
                    if np.isfinite(pitch_y)
                    else np.nan
                ),
                "X_MPLSoccer": (
                    pitch_x / CONFIG.length
                    if np.isfinite(pitch_x)
                    else np.nan
                )
            }

        # -------------------------
        # SAVE BALL
        # -------------------------
        if n_ball > 0:

            frames_with_ball += 1

            ball_xyxy = ball_detections.xyxy[0]

            ball_pitch_x = np.nan
            ball_pitch_y = np.nan

            if len(ball_pitch_xys) > 0:
                ball_pitch_x = float(
                    ball_pitch_xys[0][0]
                )
                ball_pitch_y = float(
                    ball_pitch_xys[0][1]
                )

            ball[str(frame_number)] = {
                "X1": float(ball_xyxy[0]),
                "Y1": float(ball_xyxy[1]),
                "X2": float(ball_xyxy[2]),
                "Y2": float(ball_xyxy[3]),
                "X_Pitch": ball_pitch_x,
                "Y_Pitch": ball_pitch_y,
                "Y_MPLSoccer": (
                    ball_pitch_y / CONFIG.width
                    if np.isfinite(ball_pitch_y)
                    else np.nan
                ),
                "X_MPLSoccer": (
                    ball_pitch_x / CONFIG.length
                    if np.isfinite(ball_pitch_x)
                    else np.nan
                )
            }

        else:

            ball[str(frame_number)] = {
                "X1": 0,
                "Y1": 0,
                "X2": 0,
                "Y2": 0,
                "X_Pitch": np.nan,
                "Y_Pitch": np.nan,
                "Y_MPLSoccer": np.nan,
                "X_MPLSoccer": np.nan
            }

        # -------------------------
        # ANNOTATED VIDEO
        # -------------------------
        if GENERATE_VIDEO:

            annotated_frame = frame.copy()

            if n_players > 0:

                labels = [
                    f"#{int(track_id)}"
                    for track_id in object_ids
                ]

                annotated_frame = (
                    ellipse_annotator.annotate(
                        scene=annotated_frame,
                        detections=all_detections
                    )
                )

                annotated_frame = (
                    label_annotator.annotate(
                        scene=annotated_frame,
                        detections=all_detections,
                        labels=labels
                    )
                )

            if n_ball > 0:

                annotated_frame = (
                    triangle_annotator.annotate(
                        scene=annotated_frame,
                        detections=ball_detections
                    )
                )

            sink.write_frame(
                frame=annotated_frame
            )

        frame_number += 1


# -------------------------
# SAVE CSV
# -------------------------
save_tracking_results(
    players,
    ball,
    frame_number
)

print()
print("===== ГОТОВО =====")
print("Кадров обработано:", frames_total)
print("Кадров с игроками:", frames_with_players)
print("Кадров с координатами поля:", frames_with_pitch)
print("Кадров с мячом:", frames_with_ball)

csv_path = f"./output/{VIDEO_FILE}.csv"

print("CSV:", csv_path)
print("CSV существует:", os.path.exists(csv_path))

KeyboardInterrupt: 

NameError: name 'SOURCE_VIDEO_PATH' is not defined

In [ ]:
save_tracking_results(players, ball, frame_number)